<a href="https://colab.research.google.com/github/Dara4hem/cc_/blob/main/LLM_Fine_Tune_Task_CC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Environment Setup**

Let's start by setting up our environment with all the necessary libraries.

In [1]:
# Install required packages
!pip install -q unsloth datasets huggingface_hub peft trl transformers accelerate bitsandbytes evaluate pandas scikit-learn vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.6/218.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.4/326.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from unsloth import FastLanguageModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import evaluate
import json

<ipython-input-2-25df8fb94678>:8: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-01 22:45:24 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 05-01 22:45:24 [__init__.py:239] Automatically detected platform cuda.


**2. Data Preparation**

**2.1 Loading and Exploring the Dataset**

In [3]:
# Cell 2: Load and Explore the Dataset
from google.colab import drive
import pandas as pd

#You can find uploaded file here
#https://drive.google.com/file/d/1_yW6zFHYOdNVZmUvOrkG_onP1gXqgBCd/view?usp=sharing

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Set path to the .csv file
csv_file_path = '/content/drive/MyDrive/realtor-data.csv'  # Path to your CSV file

# Step 3: Load the CSV file
df = pd.read_csv(csv_file_path)


import numpy as np
from sklearn.model_selection import train_test_split

# Load the dataset
dataset_path = '/content/drive/MyDrive/realtor-data.csv'
df = pd.read_csv(dataset_path)

# Display basic information
print(f"Dataset shape: {df.shape}")
print(df.head())
print(df.info())
print(df.describe())

Mounted at /content/drive
Dataset shape: (2226382, 12)
   brokered_by    status     price  bed  bath  acre_lot     street  \
0     103378.0  for_sale  105000.0  3.0   2.0      0.12  1962661.0   
1      52707.0  for_sale   80000.0  4.0   2.0      0.08  1902874.0   
2     103379.0  for_sale   67000.0  2.0   1.0      0.15  1404990.0   
3      31239.0  for_sale  145000.0  4.0   2.0      0.10  1947675.0   
4      34632.0  for_sale   65000.0  6.0   2.0      0.05   331151.0   

         city        state  zip_code  house_size prev_sold_date  
0    Adjuntas  Puerto Rico     601.0       920.0            NaN  
1    Adjuntas  Puerto Rico     601.0      1527.0            NaN  
2  Juana Diaz  Puerto Rico     795.0       748.0            NaN  
3       Ponce  Puerto Rico     731.0      1800.0            NaN  
4    Mayaguez  Puerto Rico     680.0         NaN            NaN  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column    

**2.2 Data Cleaning and Preprocessing**

In [5]:
# Check for missing values
print(df.isnull().sum())

# Handle missing values
df = df.dropna(subset=['price'])  # Drop rows where price is missing
df['bed'] = df['bed'].fillna(0)
df['bath'] = df['bath'].fillna(0)
df['acre_lot'] = df['acre_lot'].fillna(0)
df['house_size'] = df['house_size'].fillna(0)

# Convert price to numeric, removing any non-numeric characters
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')

# Filter any remaining invalid data
df = df[df['price'] > 0]

brokered_by         4533
status                 0
price                  0
bed                    0
bath                   0
acre_lot               0
street             10864
city                1404
state                  8
zip_code             298
house_size             0
prev_sold_date    733256
dtype: int64


**2.3 Converting Tabular Data to Text Format for LLM**

To use our structured data with an LLM, we need to convert it to a prompt-style text format.

In [7]:
def create_property_prompt(row):
    """Convert a row of tabular data into a text prompt for the LLM."""
    prompt = f"""
Property Details:
Location: {row['city']}, {row['state']}
Bedrooms: {row['bed']}
Bathrooms: {row['bath']}
Lot Size: {row['acre_lot']} acres
House Size: {row['house_size']} sq ft

Based on the above details, predict the property price.
"""
    return prompt

def create_completion(row):
    """Create the completion/target text for the LLM."""
    return f"The predicted property price is ${int(row['price']):,}."

# Apply functions to create prompts and completions
df['prompt'] = df.apply(create_property_prompt, axis=1)
df['completion'] = df.apply(create_completion, axis=1)

# Display example of prompt and completion
print("Example Prompt:")
print(df['prompt'].iloc[0])
print("\nExample Completion:")
print(df['completion'].iloc[0])

Example Prompt:

Property Details:
Location: Adjuntas, Puerto Rico
Bedrooms: 3.0
Bathrooms: 2.0
Lot Size: 0.12 acres
House Size: 920.0 sq ft

Based on the above details, predict the property price.


Example Completion:
The predicted property price is $105,000.


**2.4 Splitting the Dataset**

In [8]:
# Split data into training and evaluation sets
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

# Check sizes
print(f"Training set size: {len(train_df)}")
print(f"Evaluation set size: {len(eval_df)}")

# Create Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[['prompt', 'completion']])
eval_dataset = Dataset.from_pandas(eval_df[['prompt', 'completion']])

Training set size: 1779648
Evaluation set size: 444913


**3. Model Setup with Unsloth and LoRA**

**3.1 Base Model Selection**

In [22]:
# Define model name
# base_model_name = "unsloth/llama-3-8b"
# base_model_name = "unsloth/tiny-llama-1b"
# base_model_name = "unsloth/llama-3-8b-bnb-4bit"
# Set up quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Explanation:
# - 4-bit quantization reduces model memory by ~75% compared to 16-bit
# - nf4 (normalized float 4) has better distributional properties for LLM weights
# - Double quantization further compresses the model
# - bfloat16 compute dtype provides good numerical stability

**3.2 Unsloth and LoRA Configuration**

In [23]:
# Set up LoRA configuration
lora_config = LoraConfig(
    r=16,                     # Rank of the LoRA update matrices
    # alpha=32,                 # Alpha parameter for LoRA scaling
    lora_dropout=0.05,        # Dropout probability for LoRA layers
    task_type="CAUSAL_LM",    # The task type (causal language modeling)
    target_modules=[          # Layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",              # No bias parameters will be trained
    modules_to_save=None      # No additional modules to save
)

# Explanation:
# - r=16 provides a good balance between model capacity and efficiency
# - alpha=32 scales the LoRA update by 32/16 = 2x
# - lora_dropout=0.05 prevents overfitting
# - We target attention layers (q_proj, k_proj, v_proj, o_proj) and MLP layers (gate_proj, up_proj, down_proj)

**3.3 Initialize Model and Tokenizer with Unsloth**

In [30]:
from unsloth import FastLanguageModel
import torch

base_model_name = "unsloth/llama-2-7b-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=1024,
    dtype=torch.float16,
    load_in_4bit=True,
    device_map="auto",  # Distributes model across GPU + CPU
    # llm_int8_enable_fp32_cpu_offload=True  # ✅ ✅ this fixes the memory crash
)

# Prepare the model for LoRA fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # default LLaMA modules
    modules_to_save=[],
)


print(f"Trainable parameters: {model.get_nb_trainable_parameters()}")

==((====))==  Unsloth 2025.4.4: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

**3.4 Tokenization Function**

In [ ]:
# Function to tokenize our datasets
def tokenize_function(examples):
    # Combine prompt and completion for training
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        # Format: <|system|>You are a real estate price prediction assistant.<|user|>{prompt}<|assistant|>{completion}
        texts.append(f"<|system|>You are a real estate price prediction assistant.<|user|>{prompt}<|assistant|>{completion}")

    # Tokenize with padding and truncation
    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=2048,
        return_tensors="pt"
    )

    # Prepare the labels (set to -100 for non-prediction tokens)
    labels = tokenized["input_ids"].clone()

    # Find position of first assistant token for each example
    assistant_token_id = tokenizer.encode("<|assistant|>")[0]
    for i, input_ids in enumerate(tokenized["input_ids"]):
        # Find index of assistant token
        assistant_pos = (input_ids == assistant_token_id).nonzero(as_tuple=True)[0]
        if len(assistant_pos) > 0:
            # Set tokens before assistant token to -100 (not used for loss)
            labels[i, :assistant_pos[0]+1] = -100

    tokenized["labels"] = labels
    return tokenized

**3.5 Tokenize the Datasets**

In [ ]:
# Tokenize datasets
tokenized_train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(f"Tokenized train dataset size: {len(tokenized_train_dataset)}")
print(f"Tokenized eval dataset size: {len(tokenized_eval_dataset)}")

**4. Fine-tuning with Unsloth**

**4.1 Training Arguments**

In [ ]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results/real_estate_price_prediction",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    fp16=True,
    optim="adamw_torch",
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
)

**4.2 Trainer Setup with Unsloth**

In [ ]:
from unsloth import UnslothTrainer

# Initialize the trainer with Unsloth optimizations
trainer = UnslothTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
)

# Train the model
trainer.train()

# Save the trained model
trainer.save_model("./real_estate_price_predictor")

**5. Evaluation**

**5.1 Define Evaluation Function**

In [ ]:
def evaluate_price_predictions(model, tokenizer, eval_df):
    model.eval()
    actual_prices = []
    predicted_prices = []

    for _, row in eval_df.iterrows():
        # Format the input prompt
        prompt = f"<|system|>You are a real estate price prediction assistant.<|user|>{row['prompt']}<|assistant|>"

        # Generate prediction
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                temperature=0.1,
                do_sample=False,
            )

        # Decode the outputs
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract the predicted price using regex
        import re
        price_match = re.search(r'\$([0-9,]+)', generated_text)
        if price_match:
            try:
                predicted_price = float(price_match.group(1).replace(',', ''))
                predicted_prices.append(predicted_price)
                actual_prices.append(row['price'])
            except:
                pass  # Skip if we can't parse the price

    # Calculate metrics
    mae = mean_absolute_error(actual_prices, predicted_prices)
    mse = mean_squared_error(actual_prices, predicted_prices)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual_prices, predicted_prices)

    return {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
        'num_samples': len(actual_prices)
    }

**5.2 Run Evaluation**

In [ ]:
# Load the fine-tuned model
fine_tuned_model = FastLanguageModel.from_pretrained(
    "./real_estate_price_predictor",
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

# Evaluate the model
eval_results = evaluate_price_predictions(fine_tuned_model, tokenizer, eval_df)
print("Evaluation Results:")
print(f"MAE: ${eval_results['mae']:,.2f}")
print(f"RMSE: ${eval_results['rmse']:,.2f}")
print(f"R²: {eval_results['r2']:.4f}")
print(f"Number of samples evaluated: {eval_results['num_samples']}")

# Success threshold check
success = eval_results['mae'] < 30000  # $30k threshold
print(f"Success threshold met: {success}")

**6. Merging LoRA Adapters**

**6.1 Adapter Merging Process**


In [ ]:
# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto"
)

# Load the LoRA adapter
adapter_model = PeftModel.from_pretrained(base_model, "./real_estate_price_predictor")

# Merge the base model with the LoRA adapter
merged_model = adapter_model.merge_and_unload()

# Save the merged model
merged_model.save_pretrained("./real_estate_price_predictor_merged")
tokenizer.save_pretrained("./real_estate_price_predictor_merged")

print("Model merged and saved successfully!")

**6.2 Compare Adapter-only vs. Merged Performance**

In [ ]:
# Evaluate the adapter-only model
adapter_results = evaluate_price_predictions(adapter_model, tokenizer, eval_df.sample(50))

# Evaluate the merged model
merged_model_loaded = AutoModelForCausalLM.from_pretrained(
    "./real_estate_price_predictor_merged",
    device_map="auto"
)
merged_results = evaluate_price_predictions(merged_model_loaded, tokenizer, eval_df.sample(50))

# Compare results
print("Adapter-only model:")
print(f"MAE: ${adapter_results['mae']:,.2f}")
print(f"RMSE: ${adapter_results['rmse']:,.2f}")
print(f"R²: {adapter_results['r2']:.4f}")

print("\nMerged model:")
print(f"MAE: ${merged_results['mae']:,.2f}")
print(f"RMSE: ${merged_results['rmse']:,.2f}")
print(f"R²: {merged_results['r2']:.4f}")

print("\nPerformance should be the same or very similar. The merged model has the advantage of not requiring the adapter infrastructure.")

**7. Model Deployment with vLLM**

**7.1 Setting up vLLM for Inference**

In [ ]:
from vllm import LLM, SamplingParams

# Initialize vLLM with the merged model
vllm_model = LLM(
    model="./real_estate_price_predictor_merged",
    quantization="awq",  # Use AWQ quantization for efficient serving
    tensor_parallel_size=1,  # Set to number of GPUs for tensor parallelism
    max_model_len=2048,
    trust_remote_code=True,
)

# Define sampling parameters for inference
sampling_params = SamplingParams(
    temperature=0.1,
    top_p=0.95,
    max_tokens=50,
)

# Test inference function
def predict_price_vllm(property_details):
    """Generate price prediction using vLLM for optimized inference."""
    prompt = f"""<|system|>You are a real estate price prediction assistant.<|user|>
Property Details:
{property_details}

Based on the above details, predict the property price.<|assistant|>"""

    # Generate response
    outputs = vllm_model.generate([prompt], sampling_params)

    # Extract generated text
    generated_text = outputs[0].outputs[0].text

    # Extract price using regex
    import re
    price_match = re.search(r'\$([0-9,]+)', generated_text)

    if price_match:
        price_str = price_match.group(1).replace(',', '')
        return float(price_str)
    else:
        return None